# WORKING WITH DIFFERENT TYPES OF DATA
    We also review working with a variety of different kinds of data, including the following:

    - Booleans

    - Numbers

    - Strings

    - Dates and timestamps

    - Handling null

    - Complex types

    - User-defined functions


In [6]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('Data').getOrCreate()
df=spark.read.format('csv').load('file:///home/unik/OReily/by-day/2010-12-01.csv',header=True,inferSchema=True);

In [8]:
df.printSchema()
df.createOrReplaceTempView('dfTable')

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



# WORKING WITH BOOLEANS

In [12]:
from pyspark.sql.functions import col
df.where(col("InvoiceNo") != 536365)\
  .select("InvoiceNo", "Description")\
  .show(5, False)

+---------+-----------------------------+
|InvoiceNo|Description                  |
+---------+-----------------------------+
|536366   |HAND WARMER UNION JACK       |
|536366   |HAND WARMER RED POLKA DOT    |
|536367   |ASSORTED COLOUR BIRD ORNAMENT|
|536367   |POPPY'S PLAYHOUSE BEDROOM    |
|536367   |POPPY'S PLAYHOUSE KITCHEN    |
+---------+-----------------------------+
only showing top 5 rows



In [14]:
df.where("InvoiceNo = 536365").show(5, False)

+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |2010-12-01 08:26:00|2.55     |17850.0   |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |2010-12-01 08:26:00|2.75     |17850.0   |United Kingdom|
|536365   |84029G   |KNITTED UNION FLAG HOT WATER BOTTLE|6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
|536365   |84029E   |RED WOOLLY HOTTIE WHITE HEART.     |6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
+---------+-----

# WORKING WITH NUMBERS

In [18]:
from pyspark.sql.functions import expr,pow
fabricatedQuantity=pow(col('Quantity')*col('UnitPrice'),2)+5
df.select(expr('CustomerID'),fabricatedQuantity.alias('realQuantity')).show(5)

+----------+------------------+
|CustomerID|      realQuantity|
+----------+------------------+
|   17850.0|239.08999999999997|
|   17850.0|          418.7156|
|   17850.0|             489.0|
|   17850.0|          418.7156|
|   17850.0|          418.7156|
+----------+------------------+
only showing top 5 rows



In [19]:
df.selectExpr(
  "CustomerId",
  "(POWER((Quantity * UnitPrice), 2.0) + 5) as realQuantity").show(2)

+----------+------------------+
|CustomerId|      realQuantity|
+----------+------------------+
|   17850.0|239.08999999999997|
|   17850.0|          418.7156|
+----------+------------------+
only showing top 2 rows



In [21]:
# ROUNDING AND BROUNDING
df.select(round(lit('2.5')),bround(lit(2.5))).show(2)

+-------------+--------------+
|round(2.5, 0)|bround(2.5, 0)|
+-------------+--------------+
|          3.0|           2.0|
|          3.0|           2.0|
+-------------+--------------+
only showing top 2 rows



In [27]:
# TO FIND CORRELATION AMONG TWO VALUES
df.stat.corr('QUANTITY','UNITPRICE')
df.select(corr('QUANTITY','UNITPRICE').alias('CORRELATION')).show()

+--------------------+
|         CORRELATION|
+--------------------+
|-0.04112314436835551|
+--------------------+



In [58]:
df.describe().show();
df.select('Country').distinct().count()

+-------+-----------------+------------------+--------------------+------------------+------------------+------------------+--------------+
|summary|        InvoiceNo|         StockCode|         Description|          Quantity|         UnitPrice|        CustomerID|       Country|
+-------+-----------------+------------------+--------------------+------------------+------------------+------------------+--------------+
|  count|             3108|              3108|                3098|              3108|              3108|              1968|          3108|
|   mean| 536516.684944841|27834.304044117645|                NULL| 8.627413127413128| 4.151946589446603|15661.388719512195|          NULL|
| stddev|72.89447869788873|17407.897548583845|                NULL|26.371821677029203|15.638659854603892|1854.4496996893627|          NULL|
|    min|           536365|             10002| 4 PURPLE FLOCK D...|               -24|               0.0|           12431.0|     Australia|
|    max|          C

7

In [34]:
df.groupBy('Country','Description').agg(
    sum('Quantity').alias('Total Quantity'),\
    max('Quantity').alias('MAX QUANTITY')
).show()

+--------------+--------------------+--------------+------------+
|       Country|         Description|Total Quantity|MAX QUANTITY|
+--------------+--------------------+--------------+------------+
|United Kingdom|IVORY DINER WALL ...|             4|           4|
|          EIRE|PAINT YOUR OWN CA...|            24|          24|
|United Kingdom|VINTAGE SEASIDE J...|             1|           1|
|United Kingdom|PINK DIAMANTE PEN...|             4|           4|
|United Kingdom|YELLOW COAT RACK ...|             3|           3|
|United Kingdom|CHOCOLATE CALCULATOR|            27|          12|
|United Kingdom| LUNCH BAG CARS BLUE|            22|          10|
|United Kingdom|PAINTED METAL HEA...|             9|           4|
|United Kingdom|S/4 VALENTINE DEC...|             2|           1|
|       Germany|CHILDREN'S CIRCUS...|            12|          12|
|United Kingdom|RED DINER WALL CLOCK|             4|           4|
|United Kingdom|SMALL HANGING IVO...|            92|          72|
|United Ki

# WORKING WITH STRINGS

In [44]:
# To change the first word to UpperCase
from pyspark.sql.functions import initcap
df.select(initcap(col('Description'))).show(5)

+--------------------+
|initcap(Description)|
+--------------------+
|White Hanging Hea...|
| White Metal Lantern|
|Cream Cupid Heart...|
|Knitted Union Fla...|
|Red Woolly Hottie...|
+--------------------+
only showing top 5 rows



In [45]:
# To make all words in upper and lower 
from pyspark.sql.functions import lower, upper
df.select(col("Description"),
lower(col("Description")),
upper(lower(col("Description")))).show(5)

+--------------------+--------------------+-------------------------+
|         Description|  lower(Description)|upper(lower(Description))|
+--------------------+--------------------+-------------------------+
|WHITE HANGING HEA...|white hanging hea...|     WHITE HANGING HEA...|
| WHITE METAL LANTERN| white metal lantern|      WHITE METAL LANTERN|
|CREAM CUPID HEART...|cream cupid heart...|     CREAM CUPID HEART...|
|KNITTED UNION FLA...|knitted union fla...|     KNITTED UNION FLA...|
|RED WOOLLY HOTTIE...|red woolly hottie...|     RED WOOLLY HOTTIE...|
+--------------------+--------------------+-------------------------+
only showing top 5 rows



# REGULAR EXPRESSION
    -Probably one of the most frequently performed tasks is searching for the existence of one string in another or replacing all mentions of a string with another value this is often done by regular expressions
    

In [50]:
from pyspark.sql.functions import regexp_replace
regex_string = "BLACK|WHITE|RED|GREEN|BLUE"
df.select(regexp_replace(col("Description"),regex_string,"COLOR").alias("color_clean"),col("Description")).show(2)

+--------------------+--------------------+
|         color_clean|         Description|
+--------------------+--------------------+
|COLOR HANGING HEA...|WHITE HANGING HEA...|
| COLOR METAL LANTERN| WHITE METAL LANTERN|
+--------------------+--------------------+
only showing top 2 rows



In [52]:
from pyspark.sql.functions import translate
df.select(translate(col("Description"),"LEET","1337"),col("Description")).show(2,truncate=False)

+----------------------------------+----------------------------------+
|translate(Description, LEET, 1337)|Description                       |
+----------------------------------+----------------------------------+
|WHI73 HANGING H3AR7 7-1IGH7 HO1D3R|WHITE HANGING HEART T-LIGHT HOLDER|
|WHI73 M37A1 1AN73RN               |WHITE METAL LANTERN               |
+----------------------------------+----------------------------------+
only showing top 2 rows



In [54]:
from pyspark.sql.functions import regexp_extract
extract_str = "(BLACK|WHITE|RED|GREEN|BLUE)"
df.select(regexp_extract(col("Description"),extract_str,1).alias("color_clean"),col("Description")).show(2,truncate=False)

+-----------+----------------------------------+
|color_clean|Description                       |
+-----------+----------------------------------+
|WHITE      |WHITE HANGING HEART T-LIGHT HOLDER|
|WHITE      |WHITE METAL LANTERN               |
+-----------+----------------------------------+
only showing top 2 rows



    Note:Sometimes, rather than extracting values, we simply want to check for their existence. We can do this with the contains method on each column. This will return a Boolean declaring whether the value you specify is in the column’s string:

In [60]:
from pyspark.sql.functions import instr
containsBlack = instr(col("Description"), "BLACK") >= 1
df.withColumn("hasSimpleColor", containsBlack | containsWhite)\
  .where("hasSimpleColor")\
  .select("Description").show(3, False)

+----------------------------------+
|Description                       |
+----------------------------------+
|WHITE HANGING HEART T-LIGHT HOLDER|
|WHITE METAL LANTERN               |
|RED WOOLLY HOTTIE WHITE HEART.    |
+----------------------------------+
only showing top 3 rows



# WORKING WITH DATE AND TIMESTAMPS
    -Working with dates and timestamps closely relates to working with strings because we often store our timestamps or dates as strings and convert them into date types at runtime. 

In [61]:
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [64]:
from pyspark.sql.functions import current_date, current_timestamp
dateDF=spark.range(10)\
  .withColumn('Today',current_date())\
  .withColumn('Now',current_timestamp())
dateDF.createOrReplaceTempView('dateTable')

In [65]:
dateDF.printSchema()

root
 |-- id: long (nullable = false)
 |-- Today: date (nullable = false)
 |-- Now: timestamp (nullable = false)



In [67]:
from pyspark.sql.functions import date_add, date_sub
dateDF.select(date_sub(col("today"),5),date_add(col("today"),5)).show(1)

+------------------+------------------+
|date_sub(today, 5)|date_add(today, 5)|
+------------------+------------------+
|        2026-03-26|        2026-04-05|
+------------------+------------------+
only showing top 1 row



# WORKING WITH NULLS IN DATA
    -The primary way of interacting with null values, at DataFrame scale, is to use the .na subpackage on a DataFrame. 
    -There are two things you can do with null values: you can explicitly drop nulls or you can fill them with a value (globally or on a per-column basis). 

## COALESCE

In [70]:
from pyspark.sql.functions import coalesce
df.select(coalesce(col('Description'),col('CustomerID'))).show(5,truncate=False)

+-----------------------------------+
|coalesce(Description, CustomerID)  |
+-----------------------------------+
|WHITE HANGING HEART T-LIGHT HOLDER |
|WHITE METAL LANTERN                |
|CREAM CUPID HEARTS COAT HANGER     |
|KNITTED UNION FLAG HOT WATER BOTTLE|
|RED WOOLLY HOTTIE WHITE HEART.     |
+-----------------------------------+
only showing top 5 rows



## DROP
    -Specifying "any" as an argument drops a row if any of the values are null. Using “all” drops the row only if all values are null or NaN for that row


In [72]:
df.na.drop()
df.na.drop("any")
df.na.drop('all')

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: timestamp, UnitPrice: double, CustomerID: double, Country: string]

    - We can also apply this to certain sets of columns by passing in an array of columns:

In [73]:
df.na.drop("all",subset=["StockCode","InvoiceNo"])

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: timestamp, UnitPrice: double, CustomerID: double, Country: string]

## FILL
    -Using the fill function, you can fill one or more columns with a set of values. This can be done by specifying a map—​that is a particular value and a set of columns.

In [74]:
df.na.fill("Unknown")

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: timestamp, UnitPrice: double, CustomerID: double, Country: string]

    -We could do the same for columns of type Integer by using df.na.fill(5:Integer), or for Doubles df.na.fill(5:Double)

In [76]:
fill_cols_vals = {"StockCode":5,"Description":"No Value"}
df.na.fill(fill_cols_vals)

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: timestamp, UnitPrice: double, CustomerID: double, Country: string]

# REPLACE